# Ouroboros — train + evaluate one run

Trains one config from `configs/sweep/` (or `configs/base.yaml` with overrides) with checkpoints on Drive. **After a disconnect, just run all cells again: training resumes from the last checkpoint** (model, optimizer, scheduler, RNG and data position).

All project code runs in subprocesses (`!python ...`) so the pinned numpy etc. take effect without restarting the kernel. If the repo is private, add a Colab secret `GITHUB_TOKEN` (key icon in the left sidebar) with read access to the repository.

In [ ]:
import time, os, subprocess, json
T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true
!python --version && nproc && free -g | head -2

In [ ]:
# ---- configuration ----
REPO = 'yaniguan/ouroboros-ocsr'
BRANCH = 'claude/vigilant-johnson-j4882f'  # set to 'main' once merged
DRIVE_ROOT = '/content/drive/MyDrive/ouroboros'  # data, runs, results live here
REPO_DIR = '/content/ouroboros-ocsr'
LOCAL_DATA = '/content/data/full'  # configs expect shards at /content/data/full/shards

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for sub in ('data', 'runs', 'results', 'real'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log --oneline -1

In [ ]:
%%bash -s "$REPO_DIR"
set -e
cd "$1"
# py3nj (escnn dependency) builds from source and needs a Fortran compiler
which gfortran || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
pip install -q -r requirements-colab.txt
pip install -q --no-deps -e .
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
RUN_ID = 'A_n200k_f0_s0'     # any row of configs/sweep/index.csv
CONFIG = f'configs/sweep/{RUN_ID}.yaml'
OUT = f'{DRIVE_ROOT}/runs/{RUN_ID}'
OVERRIDES = 'train.ckpt_every=2000 data.num_workers=10'
import yaml
SIZE = yaml.safe_load(open(CONFIG))['data']['synthetic_max_samples']
print(RUN_ID, SIZE)

In [ ]:
# Copy shards Drive -> local disk (Drive FUSE is too slow for training I/O).
N_TRAIN_SHARDS = SIZE // 1000  # 1000 samples per shard; None = all
src = f'{DRIVE_ROOT}/data/full/shards'
dst = f'{LOCAL_DATA}/shards'
os.makedirs(dst, exist_ok=True)
names = sorted(os.listdir(src)) if os.path.isdir(src) else []
keep = [n for n in names if n.endswith('.tar') and (not n.startswith('train-') or
        N_TRAIN_SHARDS is None or int(n[6:12]) < N_TRAIN_SHARDS)]
t = time.time()
for n in keep:
    if not os.path.exists(f'{dst}/{n}'):
        subprocess.run(['cp', f'{src}/{n}', f'{dst}/{n}'], check=True)
print(f'{len(keep)} shards in {time.time() - t:.0f}s' if keep else 'no shards on Drive yet')
!du -sh {dst}

In [ ]:
# real-data shards (Am1-A), only needed when real_fraction > 0
!mkdir -p /content/real && rsync -a {DRIVE_ROOT}/real/ /content/real/ && ls /content/real

In [ ]:
t = time.time()
!python -m ouroboros.train --config {CONFIG} --out {OUT} --set {OVERRIDES}
TRAIN_H = (time.time() - t) / 3600

In [ ]:
t = time.time()
!python scripts/evaluate.py --run {OUT}
EVAL_H = (time.time() - t) / 3600
print(f'train {TRAIN_H:.2f} GPU-h (this session), eval {EVAL_H:.2f} GPU-h')

In [ ]:
# ---- report back: the eval printout above, and these lines ----
!tail -3 {OUT}/log.jsonl
!grep -h img_per_s {OUT}/log.jsonl | tail -1